In [ ]:
# 13 - Ensembles y Gradient Boosting

Este notebook corresponde al Rol 3: Ensemble Engineer del Sprint 4.

El objetivo es entrenar y optimizar modelos avanzados de Gradient Boosting, específicamente XGBoost y LightGBM, usando el dataset de entrenamiento procesado.

El test set no se utiliza en este notebook, ya que será reservado para la validación final del modelo seleccionado.

In [4]:
import os
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [5]:
DATA_PATH = "../data/processed/X_train_balanced_final.csv"

MODELS_DIR = "../models"
RESULTS_PATH = "../models/ensemble_results_xgb_lgbm.csv"

XGB_BASE_PATH = "../models/xgboost_base.pkl"
LGBM_BASE_PATH = "../models/lightgbm_base.pkl"
XGB_TUNED_PATH = "../models/tuned_xgboost.pkl"
LGBM_TUNED_PATH = "../models/tuned_lightgbm.pkl"

os.makedirs(MODELS_DIR, exist_ok=True)

In [6]:
df = pd.read_csv(DATA_PATH)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras columnas:")
print(df.columns.tolist()[:15])

print("\nDistribución de Revenue:")
print(df["Revenue"].value_counts())

Dimensiones del dataset: (16476, 55)

Primeras columnas:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Distribución de Revenue:
Revenue
1    8238
0    8238
Name: count, dtype: int64


In [7]:
TARGET_COL = "Revenue"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("X:", X.shape)
print("y:", y.shape)

print("\nDistribución porcentual del target:")
print(y.value_counts(normalize=True).round(3))

X: (16476, 54)
y: (16476,)

Distribución porcentual del target:
Revenue
1    0.5
0    0.5
Name: proportion, dtype: float64


In [8]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc"
}

In [9]:
def evaluate_model_cv(model, X, y, cv, scoring, model_name, version):
    """
    Evalúa un modelo usando validación cruzada estratificada.
    Devuelve métricas promedio y desviación estándar.
    """
    start_time = time.time()
    
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )
    
    elapsed_time = time.time() - start_time
    
    result = {
        "model": model_name,
        "version": version,
        "accuracy_cv_mean": scores["test_accuracy"].mean(),
        "accuracy_cv_std": scores["test_accuracy"].std(),
        "f1_cv_mean": scores["test_f1"].mean(),
        "f1_cv_std": scores["test_f1"].std(),
        "precision_cv_mean": scores["test_precision"].mean(),
        "precision_cv_std": scores["test_precision"].std(),
        "recall_cv_mean": scores["test_recall"].mean(),
        "recall_cv_std": scores["test_recall"].std(),
        "roc_auc_cv_mean": scores["test_roc_auc"].mean(),
        "roc_auc_cv_std": scores["test_roc_auc"].std(),
        "time_seconds": round(elapsed_time, 2)
    }
    
    return result

In [10]:
results = []

In [11]:
print("Todo listo para entrenar.")
print("Cantidad de filas:", X.shape[0])
print("Cantidad de variables:", X.shape[1])

Todo listo para entrenar.
Cantidad de filas: 16476
Cantidad de variables: 54


In [ ]:
## Modelo 1: XGBoost base

Se entrena un modelo XGBoost con hiperparámetros iniciales razonables.  
Este modelo servirá como punto de comparación frente a la versión tuneada.

In [12]:
xgb_base = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_base_result = evaluate_model_cv(
    model=xgb_base,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="XGBoost",
    version="base"
)

results.append(xgb_base_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,9.22


In [13]:
xgb_base.fit(X, y)

joblib.dump(xgb_base, XGB_BASE_PATH)

print(f"Modelo XGBoost base guardado en: {XGB_BASE_PATH}")

Modelo XGBoost base guardado en: ../models/xgboost_base.pkl


In [ ]:
## Modelo 2: LightGBM base

Se entrena un modelo LightGBM con hiperparámetros iniciales razonables.  
Este modelo también pertenece a la familia de Gradient Boosting y se comparará contra XGBoost.

In [14]:
lgbm_base = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

lgbm_base_result = evaluate_model_cv(
    model=lgbm_base,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="LightGBM",
    version="base"
)

results.append(lgbm_base_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,9.22
1,LightGBM,base,0.939791,0.003349,0.939847,0.003388,0.938942,0.003385,0.940761,0.004282,0.986877,0.001859,8.81


In [15]:
lgbm_base.fit(X, y)

joblib.dump(lgbm_base, LGBM_BASE_PATH)

print(f"Modelo LightGBM base guardado en: {LGBM_BASE_PATH}")

Modelo LightGBM base guardado en: ../models/lightgbm_base.pkl


In [ ]:
## Tuning de modelos

Luego de entrenar los modelos base, se aplica RandomizedSearchCV para optimizar hiperparámetros de XGBoost y LightGBM.

Se utiliza F1-score como métrica principal, ya que permite equilibrar precision y recall en un problema de clasificación binaria.